In [ ]:
!pip install -q sdv
import os
WORK = '/content'
print(sorted(f for f in os.listdir(WORK) if f.endswith('.csv')))

In [ ]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')

final = pd.read_csv(f'{WORK}/Final Data.csv', low_memory=False)
test  = pd.read_csv(f'{WORK}/test.csv', low_memory=False)[final.columns]
train = final[~final['flow_key'].isin(test['flow_key'])].reset_index(drop=True)

attack_train = train[train['is_attack'] == True].drop(columns=['is_attack']).reset_index(drop=True)
benign_train = train[train['is_attack'] == False].reset_index(drop=True)
print('attack_train:', len(attack_train), '| benign_train:', len(benign_train), '| test:', len(test))
assert len(attack_train) == 3116 and len(benign_train) == 404

ID_COLS = ['flow_key', 'timeframe_id', 'ip_a', 'ip_b', 'port_a', 'port_b']

def load_syn(path):
    s = pd.read_csv(path, low_memory=False).reindex(columns=final.columns).drop(columns=['is_attack'])
    for c in s.columns:
        if c in ID_COLS: continue
        if final[c].dtype == bool:
            s[c] = s[c].astype(str).str.strip().str.lower().map({'true':1,'false':0,'1':1,'0':0,'1.0':1,'0.0':0})
        else:
            s[c] = pd.to_numeric(s[c], errors='coerce')
    return s

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import euclidean_distances

SEED = 42
CONTEXT_PREFIX = ('source_', 'target_service_', 'target_ip_', 'flow_share_of_')

def prep_full(df):
    X = df.drop(columns=[c for c in ID_COLS + ['is_attack'] if c in df.columns]).copy()
    for c in X.select_dtypes('bool').columns: X[c] = X[c].astype(int)
    return X

def prep_hard(df):
    X = prep_full(df)
    return X.drop(columns=[c for c in X.columns if c.startswith(CONTEXT_PREFIX)])

def make_models():
    return {
        'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED, n_jobs=-1),
        'LogisticReg':  LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED),
        'SVM':          LinearSVC(C=1.0, class_weight='balanced', max_iter=5000, random_state=SEED),
        'MLP':          MLPClassifier(hidden_layer_sizes=(100,50), activation='relu', solver='adam',
                                      early_stopping=True, max_iter=500, random_state=SEED),
    }

def tstr_hard(attack_rows):
    a = attack_rows.copy(); a['is_attack'] = True
    tr = pd.concat([a, benign_train], ignore_index=True)
    Xtr, ytr = prep_hard(tr), tr['is_attack'].astype(int)
    Xte, yte = prep_hard(test)[Xtr.columns], test['is_attack'].astype(int)
    out = {}
    for name, m in make_models().items():
        pipe = make_pipeline(SimpleImputer(strategy='constant', fill_value=0, keep_empty_features=True),
                             StandardScaler(), m).fit(Xtr, ytr)
        out[name] = f1_score(yte, pipe.predict(Xte))
    return out

R  = prep_full(test[test['is_attack'] == True].drop(columns=['is_attack'])).fillna(0).astype(float)
qt = QuantileTransformer(n_quantiles=1000, output_distribution='uniform', random_state=SEED).fit(R)
Rz = qt.transform(R)
def subs(A, n, seed): return A[np.random.default_rng(seed).choice(len(A), min(n, len(A)), replace=False)]
d0 = euclidean_distances(subs(Rz, 1000, 0), squared=True)
GAMMAS = [1.0 / (np.median(d0[d0 > 0]) * f) for f in (0.1, 1, 10)]

def mmd2(X, n=1000):
    A = subs(Rz, n, 1)
    B = subs(qt.transform(prep_full(X)[R.columns].fillna(0).astype(float)), n, 2)
    K = lambda P, Q: sum(np.exp(-g * euclidean_distances(P, Q, squared=True)) for g in GAMMAS) / len(GAMMAS)
    Kxx, Kyy, Kxy = K(A, A), K(B, B), K(A, B); m, k = len(A), len(B)
    return (Kxx.sum() - np.trace(Kxx)) / (m*(m-1)) + (Kyy.sum() - np.trace(Kyy)) / (k*(k-1)) - 2 * Kxy.mean()
print('ready | real test attacks:', R.shape)

In [ ]:
import torch, random, time
from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer
from rdt.transformers.numerical import FloatFormatter

N = 3300
print('GPU:', torch.cuda.is_available())

def run_ctgan(k, seed, epochs=1000):
    path = f'{WORK}/ctgan_k{k}_seed{seed}.csv'
    if os.path.exists(path):
        print(f'  k={k} seed={seed}: already exists, loading'); return load_syn(path)

    sub_df = attack_train.sample(n=k, random_state=seed).reset_index(drop=True)
    const_cols = [c for c in sub_df.columns if sub_df[c].nunique(dropna=False) == 1]
    const_vals = {c: sub_df[c].iloc[0] for c in const_cols}
    gan_df = sub_df.drop(columns=list(set(const_cols) | {'flow_key'}))
    orig_dtypes = gan_df.dtypes

    int_cols = gan_df.select_dtypes(include=['int64', 'bool']).columns
    DISCRETE = [c for c in int_cols if gan_df[c].nunique() <= 10]
    cols = {c: ({'sdtype': 'categorical'} if c in DISCRETE
                else {'sdtype': 'numerical', 'computer_representation': 'Float'}) for c in gan_df.columns}
    metadata = Metadata.load_from_dict({'tables': {'attack': {'columns': cols}}})

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    pac = 10
    bs  = min(500, (k // pac) * pac)
    synth = CTGANSynthesizer(metadata, enforce_min_max_values=True, enforce_rounding=True,
                             epochs=epochs, batch_size=bs, pac=pac, embedding_dim=128,
                             generator_dim=(256, 256), discriminator_dim=(256, 256),
                             generator_lr=2e-4, discriminator_lr=2e-4,
                             generator_decay=1e-6, discriminator_decay=1e-6,
                             discriminator_steps=1, log_frequency=True, verbose=False,
                             cuda=torch.cuda.is_available())
    nan_cols = [c for c in gan_df.columns if gan_df[c].isna().any()]
    synth.auto_assign_transformers(gan_df)
    if nan_cols:
        synth.update_transformers({c: FloatFormatter(missing_value_generation='from_column',
                                   learn_rounding_scheme=True, enforce_min_max_values=True) for c in nan_cols})

    t0 = time.time(); synth.fit(gan_df); tt = time.time() - t0
    syn = synth.sample(num_rows=N)
    for c, v in const_vals.items(): syn[c] = v
    for c, dt in orig_dtypes.items():
        if dt.kind in 'ib' and syn[c].notna().all(): syn[c] = syn[c].round().astype(dt)
    syn['flow_key'] = (syn['timeframe_id'].astype(str) + '|' + syn['proto'].astype(str) + '|' +
                       syn['ip_a'].astype(str) + '|' + syn['ip_b'].astype(str) + '|' +
                       syn['port_a'].astype(str) + '|' + syn['port_b'].astype(str))
    syn['is_attack'] = True
    syn[final.columns].to_csv(path, index=False)
    print(f'  k={k} seed={seed}: trained in {tt/60:.1f} min (batch={bs}, {len(const_cols)} constant cols)')
    return load_syn(path)

In [ ]:
import os
print(sorted(os.listdir('/content')))

In [ ]:
K_LIST = [10, 25, 50, 100, 250, 500]
SEEDS  = [42]                          # add 43, 44 later

res_path = f'{WORK}/data_efficiency_results.csv'
done = pd.read_csv(res_path) if os.path.exists(res_path) else pd.DataFrame(columns=['k', 'seed'])

for k in K_LIST:
    for s in SEEDS:
        if ((done['k'] == k) & (done['seed'] == s)).any(): continue
        syn = run_ctgan(k, s)
        f1  = tstr_hard(syn)
        row = {'k': k, 'seed': s, **f1, 'F1_mean': np.mean(list(f1.values())), 'MMD2': mmd2(syn)}
        done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
        done.to_csv(res_path, index=False)
        print(f'k={k:4d} seed={s}  F1_mean={row["F1_mean"]:.4f}  MMD2={row["MMD2"]:.4f}')

In [ ]:
REF_FILES = {
    'CTGAN (k=3116)': 'ctgan_attack_N3300_seed42.csv',
    'Heuristic':      'heuristic_attack_N3300_seed42.csv',   # skipped if not uploaded
    'Llama':          'full_llama.csv',
    'Falcon':         'falcon_full.csv',
    'Qwen':           'qwen_full.csv',
}
ref = []
for g, f in REF_FILES.items():
    if not os.path.exists(f'{WORK}/{f}'):
        print(f'skipping {g}: {f} not found'); continue
    X = load_syn(f'{WORK}/{f}')
    if len(X) > N: X = X.sample(n=N, random_state=SEED)
    f1 = tstr_hard(X)
    ref.append({'Generator': g, **f1, 'F1_mean': np.mean(list(f1.values())), 'MMD2': mmd2(X)})
ref = pd.DataFrame(ref).set_index('Generator')
display(ref.round(4))
ref.round(4).to_csv(f'{WORK}/data_efficiency_references.csv')

In [ ]:
import matplotlib.pyplot as plt

agg  = done.groupby('k')[['F1_mean', 'MMD2']].agg(['mean', 'std'])
full = ref.loc['CTGAN (k=3116)']
ks   = list(agg.index) + [3116]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, met, lab in [(axes[0], 'F1_mean', 'TSTR-hard mean F1 (↑)'), (axes[1], 'MMD2', 'MMD² (↓)')]:
    m  = list(agg[(met, 'mean')]) + [full[met]]
    sd = list(agg[(met, 'std')].fillna(0)) + [0]
    ax.errorbar(ks, m, yerr=sd, marker='o', color='black', capsize=4, label='CTGAN trained on k real attacks')
    for g, c in [('Llama', 'tab:blue'), ('Falcon', 'tab:orange'), ('Qwen', 'tab:green')]:
        if g in ref.index: ax.axhline(ref.loc[g, met], ls='--', color=c, label=f'{g} (0 attack rows)')
    if 'Heuristic' in ref.index:
        ax.axhline(ref.loc['Heuristic', met], ls=':', color='grey', label='Heuristic (0 attack rows)')
    ax.set_xscale('log'); ax.set_xticks(ks); ax.set_xticklabels(ks)
    ax.set_xlabel('Real attack rows given to CTGAN (k)'); ax.set_ylabel(lab); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8, loc='lower right')
plt.tight_layout(); fig.savefig(f'{WORK}/data_efficiency.png', dpi=300, bbox_inches='tight'); plt.show()

cross = []
for g in [x for x in ['Llama', 'Falcon', 'Qwen'] if x in ref.index]:
    f1_k  = next((k for k in agg.index if agg.loc[k, ('F1_mean', 'mean')] >= ref.loc[g, 'F1_mean']), '>500')
    mmd_k = next((k for k in agg.index if agg.loc[k, ('MMD2', 'mean')] <= ref.loc[g, 'MMD2']), '>500')
    cross.append({'LLM': g, 'CTGAN needs k for F1': f1_k, 'CTGAN needs k for MMD²': mmd_k})
display(pd.DataFrame(cross))

In [ ]:
from google.colab import files
!zip -q -o data_efficiency.zip ctgan_k*_seed*.csv data_efficiency_results.csv data_efficiency_references.csv data_efficiency.png
files.download('data_efficiency.zip')

In [ ]:
copy_res = []
for k in K_LIST:
    base = attack_train.sample(n=k, random_state=42)
    X = base.sample(n=N, replace=True, random_state=42).reset_index(drop=True)   # pure copies
    f1 = tstr_hard(X)
    copy_res.append({'k': k, 'F1_mean': np.mean(list(f1.values())), 'MMD2': mmd2(X)})
copy_res = pd.DataFrame(copy_res).set_index('k')
display(pd.concat({'CTGAN': done.set_index('k')[['F1_mean', 'MMD2']].astype(float),
                   'Copy k rows': copy_res}, axis=1).round(4))